<a href="https://colab.research.google.com/github/Arczisork/Sztuczna-inteligencja/blob/main/RAG/RAG_Artur_11971.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q faiss-cpu pymupdf sentence-transformers tqdm ollama
!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
!nohup ollama serve > ollama.log 2>&1 &
!sleep 10
!ollama pull llama3.2:3b
!ollama list

In [ ]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import ollama

embedder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
model_id = "llama3.2:3b"

print("Embedding model załadowany")
print("Model Ollama:", model_id)

In [ ]:
import os
import json
import faiss
import pymupdf
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import ollama

embedder = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
model_id = "llama3.2:3b"

print("Embedding model załadowany")
print("Model Ollama:", model_id)

In [ ]:
import os

os.makedirs("knowledge", exist_ok=True)
os.makedirs("vec_db", exist_ok=True)

print(os.getcwd())
print(os.listdir())

In [ ]:
os.makedirs("knowledge", exist_ok=True)
os.makedirs("vec_db", exist_ok=True)

pdf_files = [
    f for f in os.listdir("knowledge")
    if f.lower().endswith(".pdf")
]

print("Pliki PDF znalezione w katalogu knowledge:")

for pdf in pdf_files:
    print("-", pdf)

if len(pdf_files) == 0:
    raise Exception(
        "Brak plików PDF w katalogu knowledge. "
        "Dodaj PDF do repozytorium GitHub w folderze knowledge."
    )

In [ ]:
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []

print("Liczba chunków:", index.ntotal)

In [ ]:
class Utils:
    def __init__(
        self,
        embedding_model=None,
        llm_model="llama3.2:3b",
        index=None,
        metadata=None,
        chunk_size=1000
    ):
        self.embedding_model = embedding_model
        self.llm_model = llm_model
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def extract_text_from_pdf(self, pdf_path):
        text = []
        pdf_document = pymupdf.open(pdf_path)

        for page_num in range(len(pdf_document)):
            page = pdf_document.load_page(page_num)
            page_text = str(page.get_text()).replace("\n", " ")
            text.append((page_num, page_text))

        return text

    def chunk_text(self, text):
        chunks = []

        for page_num, page_text in text:
            page_text = page_text.strip()

            for i in range(0, len(page_text), self.chunk_size):
                chunk = page_text[i:i+self.chunk_size].strip()

                if chunk:
                    chunks.append((page_num, chunk))

        return chunks

    def add_chunks_to_faiss(self, chunks, filename, db_loc="vec_db/"):
        os.makedirs(db_loc, exist_ok=True)

        for chunk_num, (page_number, chunk) in enumerate(
            tqdm(chunks, desc=f"Dodawanie {filename}")
        ):
            embedding = self.embedding_model.encode(
                chunk,
                show_progress_bar=False
            )

            embedding = np.array([embedding]).astype("float32")
            self.index.add(embedding)

            self.metadata.append({
                "filename": filename,
                "page_number": page_number,
                "chunk_num": chunk_num,
                "chunk": chunk
            })

        faiss.write_index(
            self.index,
            os.path.join(db_loc, "vector_database.index")
        )

        with open(
            os.path.join(db_loc, "metadata.json"),
            "w",
            encoding="utf-8"
        ) as file:
            json.dump(
                self.metadata,
                file,
                ensure_ascii=False,
                indent=2
            )

    def process_file(self, file_path):
        if not file_path.lower().endswith(".pdf"):
            print("Pomijam plik:", file_path)
            return 0

        text = self.extract_text_from_pdf(file_path)
        chunks = self.chunk_text(text)

        self.add_chunks_to_faiss(
            chunks,
            filename=os.path.basename(file_path)
        )

        return len(chunks)

    def answer_question(
        self,
        prompt_template,
        query,
        max_tokens=512,
        temp=0.1,
        k=10
    ):
        question_embedding = self.embedding_model.encode(
            query,
            show_progress_bar=False
        )

        question_embedding = np.array([question_embedding]).astype("float32")

        D, I = self.index.search(question_embedding, k)

        chunks = [
            self.metadata[i]
            for i in I[0]
            if i != -1
        ]

        context = ""

        for i, chunk in enumerate(chunks):
            context += (
                f"[{i+1}] Plik: {chunk['filename']}, "
                f"strona {chunk['page_number'] + 1}\n"
            )
            context += chunk["chunk"] + "\n\n"

        prompt = prompt_template.format(
            context=context,
            query=query
        )

        response = ollama.chat(
            model=self.llm_model,
            messages=[
                {
                    "role": "system",
                    "content": (
                        "Odpowiadaj tylko na podstawie podanego kontekstu. "
                        "Nie wymyślaj informacji. "
                        "Jeśli kontekst nie zawiera odpowiedzi, napisz: "
                        "Kontekst nie zawiera odpowiedzi na to pytanie."
                    )
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": temp,
                "num_predict": max_tokens
            }
        )

        return response["message"]["content"], chunks

In [ ]:
utils = Utils(
    embedding_model=embedder,
    llm_model=model_id,
    index=index,
    metadata=metadata,
    chunk_size=1000
)

print("RAG gotowy")

In [ ]:
knowledge_dir = "knowledge"

pdf_files = [
    f for f in os.listdir(knowledge_dir)
    if f.lower().endswith(".pdf")
]

for file in pdf_files:
    print("Przetwarzam:", file)
    utils.process_file(os.path.join(knowledge_dir, file))

print("Liczba chunków:", index.ntotal)

In [ ]:
prompt_template = """
Kontekst:
{context}

Pytanie:
{query}

Odpowiedz po polsku, krótko i konkretnie.
Korzystaj wyłącznie z podanego kontekstu.
Jeżeli odpowiedzi nie ma w kontekście, napisz:
Kontekst nie zawiera odpowiedzi na to pytanie.
"""

In [ ]:
answer, chunks = utils.answer_question(
    prompt_template=prompt_template,
    query="Zamówienie musi być wykonane w terminie do ilu miesięcy od dnia podpisania umowy?",
    k=10
)

print("ODPOWIEDŹ:")
print(answer)

In [ ]:
print("ŹRÓDŁA:")

for i, c in enumerate(chunks):
    print(f"\n--- ŹRÓDŁO {i+1} ---")
    print("Plik:", c["filename"])
    print("Strona:", c["page_number"] + 1)
    print(c["chunk"][:700])

In [ ]:
questions = [
    "Jaka była wysokość wadium?",
    "Jakie były kryteria oceny ofert?",
    "Jakie było znaczenie ceny w ocenie ofert?",
    "Jaki był termin wykonania zamówienia?",
    "Jakie instytucje miały zostać zintegrowane przez system?",
    "Jakie wymagania miał spełniać kierownik projektu?"
]

for q in questions:
    answer, chunks = utils.answer_question(
        prompt_template=prompt_template,
        query=q,
        k=10
    )

    print("=" * 80)
    print("PYTANIE:", q)
    print("ODPOWIEDŹ:", answer)
    print("ŹRÓDŁA:", [c["page_number"] + 1 for c in chunks[:3]])